# DataCo Supply Chain Analytics

## End-to-End Supply Chain Performance & Operational Efficiency

DataCo's operational records capture activity across orders, products,
customers, markets, shipping, fulfillment, and financial performance.

The objective of this analysis is to understand how these parts of the supply
chain work together and where the underlying data reveals meaningful differences
in operational performance.

The analysis will move from the overall operation into specific areas of
performance, following the evidence as patterns emerge. The focus is not only
on measuring what happened, but on understanding where performance differs,
what factors are associated with those differences, and what they mean from a
business perspective.

The final outcome will bring the strongest findings together into a practical
set of actions for improving supply chain efficiency and decision-making.

## 1. Data Foundation & Scope

The project contains three available data sources covering the core supply
chain records, dataset documentation, and access activity.

The main supply chain dataset forms the basis of the operational analysis.
The accompanying description data provides additional context for the
available fields, while the access-log data records system activity associated
with the project data.

The structure, coverage, and usability of these sources will be examined
before moving into the operational analysis.

### Data Access & SQL Setup

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("sql/supply_chain.db")

print("Supply Chain database connected.")

Supply Chain database connected.


## Overall Dataset Overview

The project brings together three data sources that provide different views of
the available information.

The main supply chain dataset contains the operational records used for the
core analysis. The dataset description provides reference information for
understanding the available fields, while the access logs provide a separate
view of system activity.

The size and coverage of each source are reviewed first, followed by a closer
look at the fields that are relevant to the supply chain analysis.

In [2]:
query = """
SELECT 'supply_chain' AS data_source, COUNT(*) AS total_rows
FROM supply_chain

UNION ALL

SELECT 'dataset_description', COUNT(*)
FROM dataset_description

UNION ALL

SELECT 'access_logs', COUNT(*)
FROM access_logs;
"""

pd.read_sql_query(query, conn)

,data_source,total_rows
0,supply_chain,180519
1,dataset_description,52
2,access_logs,469977


In [3]:
print("=== Supply Chain Dataset ===")
query = """
SELECT *
FROM supply_chain
LIMIT 5;
"""
display(pd.read_sql_query(query, conn))


print("=== Dataset Description ===")
query = """
SELECT *
FROM dataset_description
LIMIT 5;
"""
display(pd.read_sql_query(query, conn))


print("=== Access Logs ===")
query = """
SELECT *
FROM access_logs
LIMIT 5;
"""
display(pd.read_sql_query(query, conn))

=== Supply Chain Dataset ===


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,None,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,None,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,None,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,None,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,None,1360,73,None,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


=== Dataset Description ===


,FIELDS,DESCRIPTION
0,Type,: Type of transaction made
1,Days for shipping (real),: Actual shipping days of the purchased product
2,Days for shipment (scheduled),: Days of scheduled delivery of the purchased...
3,Benefit per order,: Earnings per order placed
4,Sales per customer,: Total sales per customer made per customer


=== Access Logs ===


,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...
3,Under Armour Men's Compression EV SL Slide,electronics,9/1/2017 6:00,Sep,6,footwear,206.56.112.1,/department/footwear/category/electronics/prod...
4,Pelican Sunstream 100 Kayak,water sports,9/1/2017 6:01,Sep,6,fan shop,136.108.56.242,/department/fan%20shop/category/water%20sports...


## Data Quality Check

The available data is checked for completeness and consistency before the
operational analysis begins.

Missing values and duplicate records are reviewed across the three sources,
with additional validation applied to fields where their structure requires
specific interpretation.

### Duplicate Check

In [4]:
query = """
SELECT 'supply_chain' AS data_source,
       COUNT(*) AS total_rows,
       COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM supply_chain)) AS duplicate_rows
FROM supply_chain

UNION ALL

SELECT 'dataset_description',
       COUNT(*),
       COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM dataset_description))
FROM dataset_description

UNION ALL

SELECT 'access_logs',
       COUNT(*),
       COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM access_logs))
FROM access_logs;
"""

pd.read_sql_query(query, conn)

,data_source,total_rows,duplicate_rows
0,supply_chain,180519,0
1,dataset_description,52,0
2,access_logs,469977,3249


### Duplicate Row Review

The duplicate check shows no repeated rows in the `supply_chain` and
`dataset_description` sources. The `access_logs` source contains 3,249
duplicate rows out of 469,977 records.

The duplicate records are retained in the source data at this stage and will be
considered only if they affect any analysis based on the access-log data.

### Missing Value Check

In [5]:
query = """
SELECT 'supply_chain' AS data_source,
       COUNT(*) AS total_rows,
       (COUNT(*) - COUNT("Customer Lname"))
       + (COUNT(*) - COUNT("Customer Zipcode"))
       + (COUNT(*) - COUNT("Order Zipcode"))
       + (COUNT(*) - COUNT("Product Description")) AS missing_values
FROM supply_chain

UNION ALL

SELECT 'dataset_description',
       COUNT(*),
       (COUNT(*) - COUNT(FIELDS))
       + (COUNT(*) - COUNT(DESCRIPTION))
FROM dataset_description

UNION ALL

SELECT 'access_logs',
       COUNT(*),
       (COUNT(*) - COUNT(Product))
       + (COUNT(*) - COUNT(Category))
       + (COUNT(*) - COUNT(Date))
       + (COUNT(*) - COUNT(Month))
       + (COUNT(*) - COUNT(Hour))
       + (COUNT(*) - COUNT(Department))
       + (COUNT(*) - COUNT(ip))
       + (COUNT(*) - COUNT(url))
FROM access_logs;
"""

pd.read_sql_query(query, conn)

,data_source,total_rows,missing_values
0,supply_chain,180519,336209
1,dataset_description,52,0
2,access_logs,469977,0


In [6]:
query = """
SELECT 'supply_chain' AS data_source,
       'Product Description' AS field_name,
       COUNT(*) - COUNT("Product Description") AS missing_values
FROM supply_chain

UNION ALL

SELECT 'supply_chain',
       'Order Zipcode',
       COUNT(*) - COUNT("Order Zipcode")
FROM supply_chain

UNION ALL

SELECT 'supply_chain',
       'Customer Lname',
       COUNT(*) - COUNT("Customer Lname")
FROM supply_chain

UNION ALL

SELECT 'supply_chain',
       'Customer Zipcode',
       COUNT(*) - COUNT("Customer Zipcode")
FROM supply_chain

UNION ALL

SELECT 'dataset_description',
       'All fields',
       (COUNT(*) - COUNT(FIELDS)) + (COUNT(*) - COUNT(DESCRIPTION))
FROM dataset_description

UNION ALL

SELECT 'access_logs',
       'All fields',
       (COUNT(*) - COUNT(Product))
     + (COUNT(*) - COUNT(Category))
     + (COUNT(*) - COUNT(Date))
     + (COUNT(*) - COUNT(Month))
     + (COUNT(*) - COUNT(Hour))
     + (COUNT(*) - COUNT(Department))
     + (COUNT(*) - COUNT(ip))
     + (COUNT(*) - COUNT(url))
FROM access_logs;
"""

pd.read_sql_query(query, conn)

,data_source,field_name,missing_values
0,supply_chain,Product Description,180519
1,supply_chain,Order Zipcode,155679
2,supply_chain,Customer Lname,8
3,supply_chain,Customer Zipcode,3
4,dataset_description,All fields,0
5,access_logs,All fields,0


### Missing Value Review

The missing-value check shows that incomplete data is limited to four fields in
the supply chain records.

`Product Description` has 180,519 missing values, meaning the field is
completely unavailable across the 180,519 records. `Order Zipcode` has
155,679 missing values, while `Customer Lname` and `Customer Zipcode` have
only 8 and 3 missing values respectively.

Together, these four fields account for **336,209 missing cells** in the supply
chain dataset. The `dataset_description` and `access_logs` sources do not show
missing values in the reviewed fields.

The identified incomplete fields are therefore retained for consideration only
where they are relevant to the analysis.

## Operational Baseline



A clear view of the operating scale, order-status mix, and record structure
sets the context for the supply chain analysis that follows.

### Operation Scale

In [7]:
query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT "Order Id") AS unique_orders,
    COUNT(DISTINCT "Product Name") AS unique_products,
    COUNT(DISTINCT "Customer Id") AS unique_customers
FROM supply_chain;
"""

pd.read_sql_query(query, conn)

,total_records,unique_orders,unique_products,unique_customers
0,180519,65752,118,20652


### Order Status Distribution

In [8]:
query = """
SELECT
    "Order Status" AS order_status,
    COUNT(*) AS records
FROM supply_chain
GROUP BY "Order Status"
ORDER BY records DESC;
"""

pd.read_sql_query(query, conn)

,order_status,records
0,COMPLETE,59491
1,PENDING_PAYMENT,39832
2,PROCESSING,21902
3,PENDING,20227
4,CLOSED,19616
5,ON_HOLD,9804
6,SUSPECTED_FRAUD,4062
7,CANCELED,3692
8,PAYMENT_REVIEW,1893


### Record Structure

In [9]:
query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT "Order Id") AS unique_orders,
    ROUND(
        CAST(COUNT(*) AS REAL) / COUNT(DISTINCT "Order Id"),
        2
    ) AS records_per_order
FROM supply_chain;
"""

pd.read_sql_query(query, conn)

,total_records,unique_orders,records_per_order
0,180519,65752,2.75


### Baseline Context

Understanding the operating scale helps establish the breadth of supply chain
activity represented in the data. Reviewing order-status distribution provides
an initial view of how that activity is positioned across different stages of
the order lifecycle.

Record structure matters for interpreting later metrics. With 180,519 records
covering 65,752 distinct orders, the dataset contains an average of 2.75 records
per order, making the distinction between record-level and order-level analysis
important.

These baseline checks establish the context needed to assess demand, fulfillment,
delivery, shipping, and cost performance in the stages that follow.

## End-to-End Supply Chain Performance & Operational Efficiency

The analysis follows supply chain activity from demand and order movement
through product flow, fulfillment, delivery, shipping, geographic performance,
and cost drivers, leading into the investigation of operational exceptions and
their underlying causes.

## Demand & Order Dynamics

The Order activity is examined across time to understand how demand moves through
the operating period. Changes in order volume, order size, and quantity
behaviour will be reviewed to identify meaningful demand patterns and their
potential operational implications.

### Order Volume & Movement Over Time

In [10]:
query = """
WITH yearly_movement AS (
    SELECT
        substr("order date (DateOrders)",
               instr("order date (DateOrders)", ' ') - 4, 4) AS order_year,
        COUNT(DISTINCT "Order Id") AS orders,
        SUM("Order Item Quantity") AS units
    FROM supply_chain
    GROUP BY order_year
)
SELECT
    order_year,
    orders,
    units,
    LAG(orders) OVER (ORDER BY order_year) AS previous_year_orders
FROM yearly_movement
ORDER BY order_year;
"""

pd.read_sql_query(query, conn)

,order_year,orders,units,previous_year_orders
0,2015,20904,138480,NaN
1,2016,20859,137352,20904.0
2,2017,21866,106124,20859.0
3,2018,2123,2123,21866.0


### Order Size & Quantity Behaviour

In [11]:
query = """
WITH order_quantity AS (
    SELECT
        "Order Id",
        SUM("Order Item Quantity") AS total_units
    FROM supply_chain
    GROUP BY "Order Id"
)
SELECT
    total_units,
    COUNT(*) AS orders
FROM order_quantity
GROUP BY total_units
ORDER BY total_units;
"""

pd.read_sql_query(query, conn)

,total_units,orders
0,1,14391
1,2,4517
2,3,4519
3,4,4953
4,5,5806
5,6,5140
6,7,4735
7,8,4391
8,9,4020
9,10,3348


### Demand Pattern & Operational Signal

In [12]:
query = """
SELECT
    CAST(substr("order date (DateOrders)", 1,
        instr("order date (DateOrders)", '/') - 1) AS INTEGER) AS month,
    COUNT(DISTINCT "Order Id") AS orders,
    SUM("Order Item Quantity") AS units
FROM supply_chain
WHERE "order date (DateOrders)" LIKE '%/2015 %'
   OR "order date (DateOrders)" LIKE '%/2016 %'
   OR "order date (DateOrders)" LIKE '%/2017 %'
GROUP BY month
ORDER BY month;
"""

pd.read_sql_query(query, conn)

,month,orders,units
0,1,5304,35056
1,2,4849,32273
2,3,5327,35087
3,4,5155,33684
4,5,5302,34538
5,6,5091,32405
6,7,5297,34543
7,8,5296,34390
8,9,5159,33152
9,10,5652,26129


### Demand & Order Dynamics - Overall Observation

Order activity remains relatively stable across the complete operating years,
while the volume of units moved varies more noticeably. The order-level
quantity distribution also shows that smaller orders make up a substantial
share of overall activity.

Across the monthly pattern, order volume stays comparatively consistent while
unit movement declines during the final quarter. Together, these findings
indicate that changes in demand are driven not only by the number of orders,
but also by the quantity contained within each order.

This provides an important operational context for the next stages, where
product movement, fulfillment, and delivery performance are examined.

## Product & Fulfillment Performance

Product movement is examined alongside fulfillment performance to understand
how demand translates into operational activity. Differences across products
and categories are considered together with fulfillment speed and scheduled
versus actual shipping performance.

### Product & Category Movement

In [13]:
query = """
SELECT
    "Category Name" AS category,
    COUNT(DISTINCT "Product Name") AS products,
    COUNT(DISTINCT "Order Id") AS orders,
    SUM("Order Item Quantity") AS units
FROM supply_chain
GROUP BY "Category Name"
ORDER BY units DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,category,products,orders,units
0,Cleats,2,20386,73734
1,Women's Apparel,1,17869,62956
2,Indoor/Outdoor Games,1,16623,57803
3,Cardio Equipment,2,11355,37587
4,Shop By Sport,3,10136,32726
5,Men's Footwear,1,18783,22246
6,Fishing,1,15164,17325
7,Water Sports,2,13758,15540
8,Camping & Hiking,1,12299,13729
9,Electronics,11,3061,9436


### Fulfillment Speed & Processing

In [14]:
query = """
SELECT
    ROUND(AVG("Days for shipping (real)"), 2) AS avg_actual_days,
    ROUND(AVG("Days for shipment (scheduled)"), 2) AS avg_scheduled_days,
    ROUND(
        AVG("Days for shipping (real)" - "Days for shipment (scheduled)"),
        2
    ) AS avg_schedule_gap
FROM supply_chain;
"""

pd.read_sql_query(query, conn)

,avg_actual_days,avg_scheduled_days,avg_schedule_gap
0,3.5,2.93,0.57


### Scheduled vs Actual Performance

In [15]:
query = """
SELECT
    "Delivery Status" AS delivery_status,
    ROUND(AVG("Days for shipping (real)"), 2) AS avg_actual_days,
    ROUND(AVG("Days for shipment (scheduled)"), 2) AS avg_scheduled_days
FROM supply_chain
GROUP BY "Delivery Status"
ORDER BY avg_actual_days DESC;
"""

pd.read_sql_query(query, conn)

,delivery_status,avg_actual_days,avg_scheduled_days
0,Late delivery,4.09,2.47
1,Shipping canceled,3.48,2.90
2,Shipping on time,2.98,2.98
3,Advance shipping,2.50,4.00


### Operational Impact

In [16]:
query = """
SELECT
    "Delivery Status" AS delivery_status,
    COUNT(DISTINCT "Order Id") AS orders,
    SUM("Order Item Quantity") AS units
FROM supply_chain
GROUP BY "Delivery Status"
ORDER BY orders DESC;
"""

pd.read_sql_query(query, conn)

,delivery_status,orders,units
0,Late delivery,36048,210569
1,Advance shipping,15127,88737
2,Shipping on time,11722,68285
3,Shipping canceled,2855,16488


### Product & Fulfillment Performance - Overall Observation

Product movement is concentrated across a small group of categories, with
substantial differences in the number of units handled across categories.

Fulfillment performance also shows a clear gap between planned and actual
shipping time. Late deliveries have the highest average actual shipping time,
while advance shipments are completed ahead of their scheduled duration.

From an operational perspective, late delivery represents the largest affected
group, covering 36,048 orders and 210,569 units. This makes fulfillment timing
an important area to examine further when assessing overall delivery and
shipping effectiveness.

## Delivery & Shipping Effectiveness

The Delivery outcomes and shipping choices provide a closer view of how orders
perform as they move toward customers. The focus here is on the balance between
successful, delayed, and different shipping outcomes, with service reliability
and speed considered together to identify meaningful operational differences.

### On-Time vs Late Delivery

In [17]:
query = """
SELECT
    "Delivery Status" AS delivery_status,
    COUNT(DISTINCT "Order Id") AS orders,
    ROUND(
        100.0 * COUNT(DISTINCT "Order Id")
        / SUM(COUNT(DISTINCT "Order Id")) OVER (),
        2
    ) AS order_percentage
FROM supply_chain
WHERE "Delivery Status" IN (
    'Late delivery',
    'Shipping on time',
    'Advance shipping'
)
GROUP BY "Delivery Status"
ORDER BY orders DESC;
"""

pd.read_sql_query(query, conn)

,delivery_status,orders,order_percentage
0,Late delivery,36048,57.31
1,Advance shipping,15127,24.05
2,Shipping on time,11722,18.64


### Shipping Mode Performance

In [18]:
query = """
SELECT
    "Shipping Mode" AS shipping_mode,
    COUNT(DISTINCT "Order Id") AS orders,
    AVG("Days for shipping (real)") AS avg_actual_days
FROM supply_chain
GROUP BY "Shipping Mode"
ORDER BY avg_actual_days DESC;
"""

pd.read_sql_query(query, conn)

,shipping_mode,orders,avg_actual_days
0,Standard Class,39324,3.995907
1,Second Class,12778,3.990828
2,First Class,10079,2.000000
3,Same Day,3571,0.478279


### Service & Speed Differences

In [19]:
query = """
SELECT
    "Shipping Mode" AS shipping_mode,
    "Delivery Status" AS delivery_status,
    COUNT(DISTINCT "Order Id") AS orders
FROM supply_chain
GROUP BY "Shipping Mode", "Delivery Status"
ORDER BY "Shipping Mode", orders DESC;
"""

pd.read_sql_query(query, conn)

,shipping_mode,delivery_status,orders
0,First Class,Late delivery,9602
1,First Class,Shipping canceled,477
2,Same Day,Shipping on time,1759
3,Same Day,Late delivery,1648
4,Same Day,Shipping canceled,164
5,Second Class,Late delivery,9803
6,Second Class,Shipping on time,2453
7,Second Class,Shipping canceled,522
8,Standard Class,Advance shipping,15127
9,Standard Class,Late delivery,14995


### Delivery & Shipping Effectiveness - Overall Observation

Late delivery represents the largest share of delivery outcomes, accounting for
36,048 orders or 57.31% of the completed delivery outcomes considered.
Advance shipping accounts for 15,127 orders (24.05%), while 11,722 orders
(18.64%) were shipped on time.

Shipping speed also varies considerably across modes. Standard Class records an
average actual shipping time of 4.00 days across 39,324 orders, while Second
Class averages 3.99 days across 12,778 orders. First Class is considerably
faster at 2.00 days across 10,079 orders, and Same Day averages only 0.48 days
across 3,571 orders.

Standard Class also carries the highest number of late-delivery orders at
14,995, followed by Second Class with 9,803 and First Class with 9,602.
These results point to the higher-volume shipping modes as an important area
for improving delivery reliability and reducing delays.

## Cost, profitablity & Efficiency

Cost and profitability provide the financial side of the operational picture.
Order value, discounts, and profit are considered together to see how supply
chain activity translates into economic performance and where efficiency may
be affected by cost-related factors.

### Order Economics

In [20]:
query = """
SELECT
    COUNT(DISTINCT "Order Id") AS orders,
    ROUND(SUM("Sales"), 2) AS total_sales,
    ROUND(AVG("Order Item Total"), 2) AS avg_item_value,
    ROUND(AVG("Order Item Profit Ratio"), 2) AS avg_profit_ratio
FROM supply_chain;
"""

pd.read_sql_query(query, conn)

,orders,total_sales,avg_item_value,avg_profit_ratio
0,65752,36784735.01,183.11,0.12


### Discount & Cost Patterns

In [21]:
query = """
SELECT
    ROUND(AVG("Order Item Discount"), 2) AS avg_discount,
    ROUND(AVG("Order Item Discount Rate"), 2) AS avg_discount_rate,
    ROUND(AVG("Sales"), 2) AS avg_sales,
    ROUND(AVG("Order Item Total"), 2) AS avg_item_total
FROM supply_chain;
"""

pd.read_sql_query(query, conn)

,avg_discount,avg_discount_rate,avg_sales,avg_item_total
0,20.66,0.1,203.77,183.11


### Efficiency vs Profitability

In [22]:
query = """
SELECT
    "Shipping Mode" AS shipping_mode,
    AVG("Days for shipping (real)") AS avg_actual_days,
    AVG("Order Item Profit Ratio") AS avg_profit_ratio
FROM supply_chain
GROUP BY "Shipping Mode"
ORDER BY avg_actual_days;
"""

pd.read_sql_query(query, conn)

,shipping_mode,avg_actual_days,avg_profit_ratio
0,Same Day,0.478279,0.117568
1,First Class,2.000000,0.126486
2,Second Class,3.990828,0.118427
3,Standard Class,3.995907,0.120143


### Cost, Profitability & Efficiency - Overall Observation

Across 65,752 orders, the supply chain generated total sales of 367,847,350.10,
with an average order-item value of 183.11 and an average profit ratio of 0.12.

Order items carry an average discount of 20.66, representing a discount rate
of 0.10, while the average sales value is 203.77 and the average item total
after discount is 183.11.

Shipping speed does not translate into large differences in profitability.
Same Day has the fastest average actual shipping time at 0.48 days with a
profit ratio of 0.12, while First Class reaches 2.00 days with the highest
profit ratio of 0.13. Second Class and Standard Class take about 3.99 days,
with profit ratios of 0.12.

Overall, the results suggest that improving shipping speed alone may not
materially improve profitability. Cost, discount, and operational efficiency
need to be considered together when identifying areas for improvement.

## Geographic & Market Performance

Geographic and market-level differences are considered next to understand
whether service performance is consistent across locations or concentrated in
particular markets and regions.

The focus shifts from overall operational measures to location-based patterns,
using the earlier delivery and fulfillment signals as context for identifying
where service differences are most visible.

### Market / Regional Performance

In [23]:
import pandas as pd

supply_chain_df = pd.read_csv(
    "data/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

market_performance = (
    supply_chain_df
    .groupby("Market")
    .agg(
        orders=("Order Id", "nunique"),
        units=("Order Item Quantity", "sum"),
        sales=("Sales", "sum")
    )
    .sort_values("orders", ascending=False)
)

market_performance

,orders,units,sales
Market,,,
Europe,18561,105238,1.087240e+07
Pacific Asia,17577,83680,8.273744e+06
LATAM,17181,112942,1.027761e+07
USCA,8579,56616,5.066529e+06
Africa,3854,25603,2.294453e+06


### Geographic Service Variation

In [24]:
market_service = (
    supply_chain_df
    .groupby(["Market", "Delivery Status"])
    .size()
    .unstack(fill_value=0)
)

market_service

Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
Market,,,,
Africa,2645,6340,460,2169
Europe,11604,27743,2162,8743
LATAM,12039,28044,2285,9226
Pacific Asia,9473,22712,1675,7400
USCA,5831,14138,1172,4658


### Concentration of Operational Issues

In [25]:
late_delivery = (
    market_service["Late delivery"]
    / market_service.sum(axis=1)
    * 100
).round(2)

late_delivery = late_delivery.reset_index()
late_delivery.columns = ["Market", "Late Delivery %"]

late_delivery

,Market,Late Delivery %
0,Africa,54.59
1,Europe,55.21
2,LATAM,54.36
3,Pacific Asia,55.05
4,USCA,54.80


### Geographic & Market Performance - Overall Observation

Market activity is concentrated mainly across Europe, Pacific Asia, and LATAM,
with 18,561, 17,577, and 17,181 orders respectively. USCA and Africa account
for smaller operating volumes at 8,579 and 3,854 orders.

Late-delivery share remains high across every market, ranging from 54.36% in
LATAM to 55.21% in Europe. The relatively narrow range suggests that delivery
issues are not isolated to a single market.

Geographic differences therefore appear less important than the overall
delivery pattern at this stage. This makes it useful to look beyond location
and investigate which operational factors are associated with these recurring
exceptions.

## Exception & Bottleneck Investigation

The earlier results show that delivery issues remain widespread across markets
and shipping modes. The next step is to bring these signals together and narrow
down where operational exceptions are concentrated.

Rather than treating every variation as a separate issue, the focus here is on
finding the areas where multiple indicators point towards the same operational
bottleneck.

### Exception Concentration

In [26]:
exception_concentration = (
    supply_chain_df[
        supply_chain_df["Delivery Status"] == "Late delivery"
    ]
    .groupby(["Market", "Shipping Mode"])
    .size()
    .reset_index(name="late_orders")
    .sort_values("late_orders", ascending=False)
    .reset_index(drop=True)
)

exception_concentration.head(15)

,Market,Shipping Mode,late_orders
0,LATAM,Standard Class,11790
1,Europe,Standard Class,11304
2,Pacific Asia,Standard Class,9459
3,Europe,Second Class,7596
4,Europe,First Class,7562
5,LATAM,Second Class,7510
6,LATAM,First Class,7471
7,Pacific Asia,Second Class,6293
8,Pacific Asia,First Class,5991
9,USCA,Standard Class,5774


### Cross-Factor Investigation

In [27]:
cross_factor = (
    supply_chain_df
    .drop_duplicates("Order Id")
    .groupby(["Market", "Shipping Mode"])
    .agg(
        orders=("Order Id", "nunique"),
        late_orders=("Delivery Status", lambda x: (x == "Late delivery").sum()),
        avg_actual_days=("Days for shipping (real)", "mean")
    )
    .reset_index()
)

cross_factor["late_delivery_pct"] = (
    cross_factor["late_orders"] / cross_factor["orders"] * 100
).round(2)

cross_factor.sort_values(
    "late_delivery_pct",
    ascending=False
).reset_index(drop=True).head(15)

,Market,Shipping Mode,orders,late_orders,avg_actual_days,late_delivery_pct
0,Africa,First Class,557,536,2.000000,96.23
1,Europe,First Class,2878,2753,2.000000,95.66
2,Pacific Asia,First Class,2668,2543,2.000000,95.31
3,USCA,First Class,1338,1272,2.000000,95.07
4,LATAM,First Class,2638,2498,2.000000,94.69
5,Pacific Asia,Second Class,3469,2680,4.008071,77.26
6,Africa,Second Class,715,551,3.984615,77.06
7,Europe,Second Class,3609,2773,4.009698,76.84
8,USCA,Second Class,1678,1289,3.992253,76.82
9,LATAM,Second Class,3307,2510,3.994255,75.90


### Actual Bottleneck Validation

In [28]:
bottleneck = (
    supply_chain_df
    .groupby("Shipping Mode")["Late_delivery_risk"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index(name="Late Delivery Risk %")
)

bottleneck

,Shipping Mode,Late Delivery Risk %
0,First Class,95.32
1,Second Class,76.63
2,Same Day,45.74
3,Standard Class,38.07


### Exception & Bottleneck Investigation - Overall Observation

The Late deliveries are concentrated in specific shipping-mode and market combinations, showing that delivery performance is not consistent across the operation.

Standard Class generates the largest number of late orders because it handles a much higher volume of activity. First Class stands out differently, with the highest late-delivery risk at 95.32%.

Looking across both market and shipping mode reveals a stronger operational signal: certain combinations carry substantially higher delivery risk. These combinations provide the clearest areas for further root-cause investigation.

## Digital Demand Signals

Access activity is reviewed as a supporting signal to understand how digital
activity changes over time and which products or categories receive more
attention.

The useful patterns will then be compared with the operational data to see
whether digital activity has any relevance to supply chain movement.

### Access Activity Over Time

In [29]:
access_logs_df = pd.read_csv(
    "data/tokenized_access_logs.csv",
    encoding="latin1"
)

access_logs_df["Date"] = pd.to_datetime(access_logs_df["Date"])

access_activity = (
    access_logs_df
    .groupby(access_logs_df["Date"].dt.to_period("M"))
    .size()
    .reset_index(name="access_count")
)

access_activity

,Date,access_count
0,2017-09,137238
1,2017-10,84205
2,2017-11,80860
3,2017-12,84093
4,2018-01,83581


### Product / Category Activity

In [30]:
category_activity = (
    access_logs_df
    .groupby("Category")
    .size()
    .reset_index(name="access_count")
    .sort_values("access_count", ascending=False)
    .reset_index(drop=True)
)

category_activity.head(15)

,Category,access_count
0,cleats,27878
1,shop by sport,26227
2,featured shops,26200
3,women's apparel,25627
4,men's footwear,25241
5,girls' apparel,24581
6,electronics,20845
7,indoor outdoor games,16194
8,water sports,16186
9,hunting & shooting,15645


### Operational Relevance

In [31]:
product_access = (
    access_logs_df
    .groupby("Product")
    .size()
    .reset_index(name="access_count")
)

product_movement = (
    supply_chain_df
    .groupby("Product Name")["Order Item Quantity"]
    .sum()
    .reset_index(name="units")
)

product_comparison = product_access.merge(
    product_movement,
    left_on="Product",
    right_on="Product Name",
    how="inner"
)

product_comparison = (
    product_comparison
    .sort_values("access_count", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

product_comparison

,Product,access_count,Product Name,units
0,Perfect Fitness Perfect Rip Deck,27878,Perfect Fitness Perfect Rip Deck,73698
1,Nike Men's Dri-FIT Victory Golf Polo,25627,Nike Men's Dri-FIT Victory Golf Polo,62956
2,Nike Men's CJ Elite 2 TD Football Cleat,25241,Nike Men's CJ Elite 2 TD Football Cleat,22246
3,O'Brien Men's Neoprene Life Vest,16194,O'Brien Men's Neoprene Life Vest,57803
4,Pelican Sunstream 100 Kayak,16186,Pelican Sunstream 100 Kayak,15500
5,Diamondback Women's Serene Classic Comfort Bi,15521,Diamondback Women's Serene Classic Comfort Bi,13729
6,Field & Stream Sportsman 16 Gun Fire Safe,15178,Field & Stream Sportsman 16 Gun Fire Safe,17325
7,Under Armour Hustle Storm Medium Duffle Bag,13752,Under Armour Hustle Storm Medium Duffle Bag,846
8,Columbia Men's PFG Anchor Tough T-Shirt,13716,Columbia Men's PFG Anchor Tough T-Shirt,928
9,Nike Men's Free 5.0+ Running Shoe,13641,Nike Men's Free 5.0+ Running Shoe,36680


### Digital Demand Signals - Overall Observation

Access activity shows a noticeable peak in September 2017, followed by a lower and relatively stable level through the remaining periods.

Product-level activity also shows that digital attention is uneven across categories and products. However, higher access does not consistently correspond to higher unit movement. Some products receive substantial attention while showing comparatively low movement, while others combine strong activity with much higher unit volumes.

Digital activity therefore works best as a supporting signal for the supply chain analysis. It can highlight products that deserve further investigation, but access volume alone is not enough to explain operational performance.

## Evidence-Based Hypotheses

The findings gathered across demand, product movement, fulfillment, delivery,
shipping, geography, cost, operational exceptions, and digital activity now
provide a broader view of the supply chain.

These signals are brought together to form a small set of focused hypotheses.
Each hypothesis will be tested against the available evidence to determine
which patterns remain meaningful when the different operational factors are
considered together.

### H 1 - Shipping Mode and Late Delivery

First Class shows a much higher late-delivery risk than the other shipping
modes. This raises the question of whether shipping mode is consistently
associated with late delivery across the operation.

In [32]:
shipping_risk = (
    supply_chain_df
    .groupby("Shipping Mode")["Late_delivery_risk"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .reset_index(name="Late Delivery Risk %")
)

shipping_risk

,Shipping Mode,Late Delivery Risk %
0,First Class,95.32
1,Second Class,76.63
2,Same Day,45.74
3,Standard Class,38.07


The result supports the hypothesis. Late-delivery risk varies substantially
across shipping modes, with First Class showing the highest risk at 95.32%.
Shipping mode is therefore a meaningful factor to consider when investigating
delivery performance.

### H2 - Shipping Mode and Market

First Class has the highest late-delivery risk in the overall operation. The
next question is whether this pattern remains similar across different markets,
or whether the relationship changes by market.

In [33]:
market_shipping_risk = (
    supply_chain_df
    .groupby(["Market", "Shipping Mode"])["Late_delivery_risk"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="Late Delivery Risk %")
)

market_shipping_risk

,Market,Shipping Mode,Late Delivery Risk %
0,Africa,First Class,96.99
1,Africa,Same Day,44.01
2,Africa,Second Class,77.40
3,Africa,Standard Class,38.21
4,Europe,First Class,95.82
5,Europe,Same Day,46.43
6,Europe,Second Class,77.03
7,Europe,Standard Class,38.01
8,LATAM,First Class,94.74
9,LATAM,Same Day,48.04


The pattern remains consistent across markets. First Class records the highest
late-delivery risk in every market, while Standard Class remains substantially
lower.

This strengthens the earlier finding that shipping mode is meaningfully
associated with late delivery, rather than the pattern being limited to a
specific market.

### H3 - Order Quantity and Late Delivery

Earlier demand analysis showed that order quantities vary across the operation.
This raises a further question: whether larger orders are associated with a
higher risk of late delivery.

The relationship will be tested by comparing order-level quantity with
late-delivery risk.

In [34]:
order_quantity_risk = (
    supply_chain_df
    .groupby("Order Item Quantity")["Late_delivery_risk"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="Late Delivery Risk %")
)

order_quantity_risk

,Order Item Quantity,Late Delivery Risk %
0,1,54.88
1,2,54.34
2,3,55.12
3,4,54.82
4,5,54.80


The result does not support the hypothesis. Late-delivery risk remains
relatively stable across different order item quantities, with only small
differences between quantity levels.

This suggests that order item quantity alone is not a strong indicator of
late-delivery risk in the available data. Other operational factors are more
likely to explain the differences observed in delivery performance.

### H4 - Fulfillment Delay and Late Delivery

Earlier fulfillment analysis compared scheduled and actual shipping time. This
raises a further question: whether orders taking longer than their scheduled
shipping time are more likely to be associated with late delivery.

The relationship will be tested by comparing the actual shipping time with the
scheduled shipping time.

In [35]:
fulfillment_risk = (
    supply_chain_df
    .assign(
        shipping_delay=supply_chain_df["Days for shipping (real)"]
        - supply_chain_df["Days for shipment (scheduled)"]
    )
    .groupby("shipping_delay")["Late_delivery_risk"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="Late Delivery Risk %")
)

fulfillment_risk

,shipping_delay,Late Delivery Risk %
0,-2,0.00
1,-1,0.00
2,0,0.00
3,1,95.56
4,2,95.94
5,3,96.03
6,4,95.90


The result strongly supports the hypothesis. Orders that take longer than the
scheduled shipping time show a late-delivery risk of around 96%, while orders
that meet or beat the schedule show no late-delivery risk in the available data.

This makes fulfillment delay a strong operational signal for late delivery and
points to schedule adherence as an important area for further investigation.


## Validation & Refinement

The four hypotheses brought forward different signals around delivery
performance. Their results are now considered together to separate the findings
that remain consistent from those that need a narrower interpretation.

The focus is on retaining only the relationships supported by the evidence
before moving towards the underlying operational cause.

### Validation of Existing Hypotheses

In [36]:
validation = pd.DataFrame({
    "Hypothesis": [
        "H1 - Shipping Mode → Late Delivery",
        "H2 - Shipping Mode + Market",
        "H3 - Order Quantity → Late Delivery",
        "H4 - Fulfillment Delay → Late Delivery"
    ],
    "Finding": [
        "Strong association",
        "Pattern remains consistent across markets",
        "Weak association",
        "Strong association"
    ],
    "Assessment": [
        "Supported",
        "Supported",
        "Not Supported",
        "Strongly Supported"
    ]
})

validation

,Hypothesis,Finding,Assessment
0,H1 - Shipping Mode → Late Delivery,Strong association,Supported
1,H2 - Shipping Mode + Market,Pattern remains consistent across markets,Supported
2,H3 - Order Quantity → Late Delivery,Weak association,Not Supported
3,H4 - Fulfillment Delay → Late Delivery,Strong association,Strongly Supported


The four hypotheses bring together the strongest signals identified across
the supply chain analysis.

Shipping mode shows a clear relationship with late-delivery risk, and the
pattern remains consistent across markets. Order item quantity shows only a
weak relationship with late delivery, so it does not appear to be a major
driver in the available data.

Fulfillment delay provides the strongest operational signal, with orders
exceeding their scheduled shipping time showing substantially higher
late-delivery risk.

Together, these results narrow the investigation towards shipping-mode
performance and schedule adherence as the most meaningful areas for
understanding delivery problems.

## Refinement of Findings

The validation results help narrow the earlier observations into a clearer
operational picture.

Shipping mode remains an important signal, with First Class showing the highest
late-delivery risk across markets. Order quantity does not show a meaningful
relationship with delivery risk, so it has limited value as an explanation for
the delays observed.

Fulfillment delay stands out as the strongest signal. The evidence therefore
points towards schedule adherence and shipping-mode performance as the areas
most closely connected with late delivery.

These refined findings will be carried forward into the root-cause analysis.

## Root-Cause Synthesis

The validated findings now provide a clearer basis for understanding the
operational cause behind late delivery.

Shipping-mode performance and fulfillment delay remain the strongest signals
identified through the analysis. Their consistency across markets also suggests
that the issue has an operational pattern extending across the wider supply
chain.

The next step is to bring these signals together and identify the strongest
drivers that should guide the final root-cause interpretation.

### Strongest Operational Drivers

The validated evidence points to two dominant operational signals: fulfillment
delay and shipping-mode performance.

Fulfillment delay shows the strongest connection with late delivery, with orders
exceeding their scheduled shipping time carrying around 96% late-delivery risk.
Shipping mode also shows a substantial difference, with First Class recording
the highest risk at 95.32% across the operation.

The shipping-mode pattern remains consistent across markets, strengthening the
finding that the issue is not limited to a single geographic area. Order
quantity, however, shows only small differences in late-delivery risk and does
not provide a strong explanation for the observed delays.

Together, the evidence identifies schedule adherence and shipping-mode
performance as the strongest operational drivers associated with late delivery.

### Root-Cause Conclusion

The combined evidence points to fulfillment delay and shipping-mode performance
as the strongest operational signals associated with late delivery.

Schedule adherence shows the clearest connection with delivery risk, while
First Class consistently records high late-delivery risk across markets. Order
quantity and geographic differences provide weaker explanations for the overall
pattern.

The findings suggest that improving schedule adherence and reviewing
shipping-mode execution should be the primary areas of attention when addressing
late-delivery performance.

## Cross - Domain Findings

The analysis has identified several important signals across different parts of
the supply chain. Bringing these findings together helps distinguish isolated
patterns from relationships that reinforce the same operational concern.

The strongest connections are considered across demand, fulfillment, delivery,
shipping, geography, and cost to form a broader view of supply chain
performance.

Only the relationships supported by multiple findings will be carried forward
into the final business recommendations.

### Key Cross-Domain Relationships

Demand remains relatively stable in order volume, while unit movement changes
across periods. Product activity is also concentrated across selected
categories, creating different levels of operational workload.

Against this background, delivery performance shows a much stronger concern.
Late delivery affects 36,048 orders, while fulfillment delay and shipping-mode
performance show the clearest connection with delivery risk.

Geographic analysis adds another layer to the finding. Late-delivery levels
remain high across all markets, while the shipping-mode pattern remains
consistent across those markets.

Together, demand, fulfillment, delivery, shipping, and geographic evidence
point towards an operational performance issue that is broad across the supply
chain, with schedule adherence and shipping-mode execution standing out as the
strongest connected signals.

### Operational Finding

The combined findings show that delivery reliability is the main operational
challenge across the supply chain.

Order activity and product movement create the operating workload, while
fulfillment and shipping performance determine how reliably that workload moves
through the delivery process. Late delivery remains high across markets, and
the strongest risk signals are linked to schedule adherence and shipping-mode
performance.

Cost analysis also shows that faster shipping does not automatically translate
into higher profitability. This means operational improvements should focus on
delivery reliability and efficient execution while maintaining cost discipline.

Overall, the supply chain shows a broad delivery-performance issue with clear
opportunities to improve schedule adherence, shipping-mode execution, and
operational efficiency.

## Business Action Plan

The analysis has narrowed the main operational concerns to delivery reliability,
schedule adherence, and shipping-mode performance.

The next step is to translate these findings into practical actions that can
support operational improvement and ongoing decision-making.

Priority will be given to actions that address the strongest evidence while
also considering cost discipline and the need for continued monitoring.

### Priority Issues

The overall analysis points to delivery reliability as the primary operational
challenge across the supply chain.

Schedule adherence is the highest priority, as fulfillment delay shows the
strongest connection with late-delivery risk. Shipping-mode performance is the
next priority, with First Class showing 95.32% late-delivery risk and the
pattern remaining consistent across markets.

Late delivery also affects a substantial volume of activity, with 36,048
orders identified as late. Higher-volume shipping modes therefore deserve
attention because improving their performance can influence a larger number of
orders.

Cost efficiency should remain part of the decision-making process, since faster
shipping does not automatically translate into higher profitability.

Order quantity and geographic differences show weaker explanatory value, while
digital activity remains a supporting signal rather than a primary operational
issue.

### Recommended Actions

Improve schedule adherence by reviewing the gap between planned and actual
shipping time and identifying where delays begin in the fulfillment process.

Review First Class execution across markets, with particular attention to the
operational conditions contributing to its high late-delivery risk.

Prioritize improvement efforts in higher-volume shipping modes so that service
gains can benefit a larger number of orders.

Balance delivery improvements with cost discipline, since faster shipping does
not automatically produce higher profitability.

Use digital activity and demand patterns as supporting signals for identifying
products or periods that may require closer operational attention.

### What to Monitor / Investigate

Monitor schedule adherence regularly by comparing planned and actual shipping
time across shipping modes and markets.

Track late-delivery risk by shipping mode, with particular attention to First
Class and higher-volume modes.

Monitor delivery performance alongside profitability to ensure service
improvements do not create unnecessary cost pressure.

Continue reviewing demand patterns and digital activity as supporting signals for
changes in product or operational activity.

Investigate the operational conditions behind repeated fulfillment delays,
especially where multiple performance indicators show the same pattern.

## Executive Summary

The analysis provides an end-to-end view of supply chain performance across
demand, product movement, fulfillment, delivery, shipping, geography, cost,
and operational exceptions.

The strongest evidence points to delivery reliability as the central operational
challenge, with schedule adherence and shipping-mode performance emerging as
the most important signals.

The following summary brings together the final operational picture, strongest
findings, business impact, and priority actions for decision-making.

### Final Operational Picture

The supply chain handles 65,752 orders across multiple markets, products, and
shipping modes, with delivery performance emerging as the central operational
challenge.

Order activity remains relatively stable across the main operating years, while
unit movement varies across periods and product categories. Fulfillment also
shows a gap between scheduled and actual shipping time, with an average actual
shipping time of 3.50 days against 2.93 scheduled days.

Delivery performance is the clearest area of concern, with 36,048 orders
classified as late. Late-delivery levels remain high across markets, while
shipping-mode performance shows substantial differences in delivery risk.

Overall, the operation shows a broad delivery-reliability issue, with schedule
adherence and shipping-mode performance emerging as the strongest operational
signals.

### Strongest Findings

Delivery reliability is the strongest concern identified across the analysis.

Late delivery affects 36,048 orders, while fulfillment delay shows the clearest
connection with delivery risk. Orders exceeding their scheduled shipping time
carry around 96% late-delivery risk.

Shipping mode is another strong signal, with First Class recording 95.32%
late-delivery risk and the pattern remaining consistent across markets. Order
quantity shows only small differences in risk, so it does not provide a strong
explanation for the delays observed.

Cost analysis also shows that faster shipping does not automatically produce
higher profitability, making operational improvement and cost discipline
important to consider together.

### Business Impact

Late delivery creates a direct service impact by affecting a substantial
number of orders across the supply chain. With 36,048 orders identified as
late, delivery reliability becomes an important factor in overall operational
performance.

Fulfillment delays can also affect customer service and planning reliability
when actual shipping time exceeds the scheduled commitment. The consistently
high risk observed in First Class further indicates that shipping-mode
execution needs closer operational attention.

At the same time, faster shipping does not automatically improve
profitability. Delivery improvements therefore need to balance service
reliability, operational efficiency, and cost discipline.

Overall, improving schedule adherence and shipping execution can help reduce
delivery risk while supporting more reliable and efficient supply chain
operations.

### Priority Actions

Improve schedule adherence by identifying where actual shipping time exceeds
the planned schedule and addressing the operational conditions behind those
delays.

Review First Class execution across markets, with particular attention to the
factors contributing to its high late-delivery risk.

Prioritize improvement efforts in higher-volume shipping modes to reduce delays
across a larger number of orders.

Balance delivery improvements with cost discipline, since faster shipping does
not automatically result in higher profitability.

Continue using demand patterns and digital activity as supporting signals for
products or periods that require closer operational attention.
